# ARIMA

Owner: _(name)_

Rules shared by all three model notebooks (brief, sections 3 to 5):

- read the data and the split from `src` (never re-download, never re-split);
- fit on train, choose settings on validation, touch test once at the end;
- state the rolling forecast procedure: each forecast uses only information available at that time;
- compare several plausible configurations and keep a table of settings and validation results;
- time the training (computational cost) and save test predictions with
  `common.save_predictions("arima", df, predictions, settings=..., train_seconds=...)`, which
  converts them to index points for part 3.

### Setup (same cell in every notebook)

Runs in Colab and in Cursor. In Colab it clones the repository first; for a private repository
paste a GitHub token when asked (Settings > Developer settings > Fine-grained tokens, read access
to this repository is enough).

In [ ]:
import sys, pathlib

REPO_URL = "https://github.com/Makkaii7/sp500-forecasting.git"   # <- set once after the repo exists

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not pathlib.Path("sp500-forecasting").exists() and not pathlib.Path("src/config.py").exists():
    from getpass import getpass
    token = getpass("GitHub token (leave empty if the repo is public): ")
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    import subprocess
    subprocess.run(["git", "clone", "--quiet", url], check=True)
if IN_COLAB and pathlib.Path("sp500-forecasting").exists():
    import os
    os.chdir("sp500-forecasting")

root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config, common

df = common.load_prices()
y = common.target_series(df)
train, val, test = common.split(y)
print(root)
print(df.index.min().date(), "to", df.index.max().date(), "|", len(df), "trading days")
print(f"train {train.index.min().date()} to {train.index.max().date()} ({len(train)}) | validation {val.index.min().date()} to {val.index.max().date()} ({len(val)}) | test {test.index.min().date()} to {test.index.max().date()} ({len(test)})")
df.tail(3)

## 1. Choose p, d, q

Justify with the stationarity results from part 2 (d), the ACF and PACF, the information criteria and
the validation error. `auto_arima` runs the lecture's stepwise search (KPSS for d, then AICc for p and q);
the manual candidates next to it show that the choice was examined, not just accepted.

In [ ]:
import pmdarima as pm
with common.Stopwatch() as sw_auto:
    auto = pm.auto_arima(train.to_numpy(), seasonal=False, stepwise=True, suppress_warnings=True, information_criterion="aicc", trace=True)
print(auto.summary())
print("search time (s):", round(sw_auto.seconds, 1))

## 2. Rolling one-step forecasts (validation, then test)

Procedure: the model is fitted on the training period. Each day it forecasts the next value, then is
told the true value (`update`, which keeps the parameters and extends the sample) before forecasting
again. Nothing after the forecast date is used.

In [ ]:
def walk_forward(model, series):
    preds = []
    for t, value in series.items():
        preds.append(float(np.asarray(model.predict(n_periods=1))[0]))
        model.update([value])
    return pd.Series(preds, index=series.index)

candidates = {"auto": auto.order}          # add manual orders here, e.g. "ar1": (1, 0, 0), "ma1": (0, 0, 1)
table = []
for name, order in candidates.items():
    with common.Stopwatch() as sw:
        m = pm.ARIMA(order=order, suppress_warnings=True).fit(train.to_numpy())      # arrays: statsmodels cannot extend an irregular date index
    val_pred = common.to_price(walk_forward(m, val), df)
    table.append({"name": name, "order": order, "AICc": m.aicc(), "val RMSE (index points)": common.rmse(common.actual_price(val.index, df), val_pred), "fit seconds": sw.seconds})
pd.DataFrame(table).set_index("name").round(3)

In [ ]:
best_order = auto.order                     # or the manual order that won on validation
with common.Stopwatch() as sw:
    model = pm.ARIMA(order=best_order, suppress_warnings=True).fit(pd.concat([train, val]).to_numpy())   # refit on train + validation
test_pred = walk_forward(model, test)
test_price = common.to_price(test_pred, df)
print("test MAE  (index points)", round(common.mae(common.actual_price(test.index, df), test_price), 2))
print("test RMSE (index points)", round(common.rmse(common.actual_price(test.index, df), test_price), 2))
common.save_predictions("arima", df, test_pred, settings={"order": best_order, "refit": "train + validation, then update() each day"}, train_seconds=sw.seconds)

## 3. Residual check (the workflow's step 4)

In [ ]:
resid = (common.actual_price(test.index, df) - test_price).dropna()
from statsmodels.graphics.tsaplots import plot_acf
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
resid.plot(ax=ax[0], title="residuals over time (index points)")
plot_acf(resid, lags=30, ax=ax[1], title="ACF of residuals (should sit inside the limits)")
plt.tight_layout()